You have a table name – user_activity and you have columns user_id and activity_date
Table structure:

(1, '2024-01-01'),
(2, '2024-01-01'),
(1, '2024-01-02'),
(3, '2024-01-02'),
(2, '2024-01-02')


Problem -  Find number of active users per day


In [0]:
%sql
CREATE TABLE user_activity (
    user_id INT,
    activity_date DATE
);

In [0]:
%sql
INSERT INTO user_activity (user_id, activity_date)
VALUES
(1, '2024-01-01'),
(2, '2024-01-01'),
(1, '2024-01-02'),
(3, '2024-01-02'),
(2, '2024-01-02');


num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
SELECT * from user_activity;

user_id,activity_date
1,2024-01-01
2,2024-01-01
1,2024-01-02
3,2024-01-02
2,2024-01-02


2: Retention (Next Day Login)
Problem – 
Find users who logged in on consecutive days

In [0]:
%sql
SELECT DISTINCT user_id
FROM (
    SELECT 
        user_id,
        activity_date,
        LEAD(activity_date) OVER (
            PARTITION BY user_id 
            ORDER BY activity_date
        ) AS next_login
    FROM user_activity
) t
WHERE DATEDIFF(next_login, activity_date) = 1;

user_id
1
2


CASE STUDY 2: 


1: Top Product per Category
You have a table name is sales with column name category, product and revenue


Table structure:


('Electronics', 'Phone', 1000),('Electronics', 'Laptop', 2000),('Electronics', 'Tablet', 1500),('Clothing', 'Shirt', 500),('Clothing', 'Jacket', 800)




Problem – Find highest selling product in each category.


In [0]:
%sql
CREATE TABLE sales (
    category STRING,
    product STRING,
    revenue INT
);

In [0]:
%sql
INSERT INTO sales (category, product, revenue)
VALUES
('Electronics', 'Phone', 1000),
('Electronics', 'Laptop', 2000),
('Electronics', 'Tablet', 1500),
('Clothing', 'Shirt', 500),
('Clothing', 'Jacket', 800);

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
select * from sales;

category,product,revenue
Electronics,Phone,1000
Electronics,Laptop,2000
Electronics,Tablet,1500
Clothing,Shirt,500
Clothing,Jacket,800


In [0]:
%sql
SELECT category, product, revenue
FROM (
    SELECT 
        category,
        product,
        revenue,
        ROW_NUMBER() OVER (
            PARTITION BY category
            ORDER BY revenue DESC
        ) AS rn
    FROM sales
) t
WHERE rn = 1;

category,product,revenue
Clothing,Jacket,800
Electronics,Laptop,2000



CASE STUDY 3: 
1: Latest Transaction per User
You have table transactions with column name – user_id(int), txn_date (date) and amount(int)


Table structure:


(1, '2024-01-01', 100),(1, '2024-02-01', 200),(2, '2024-01-05', 300)


Problem - Find latest transaction for each user

In [0]:
%sql
CREATE TABLE transactions (
    user_id INT,
    txn_date DATE,
    amount INT
);

In [0]:
%sql
INSERT INTO transactions (user_id, txn_date, amount)
VALUES
(1, '2024-01-01', 100),
(1, '2024-02-01', 200),
(2, '2024-01-05', 300);

num_affected_rows,num_inserted_rows
3,3


In [0]:
%sql
select * from transactions;

user_id,txn_date,amount
1,2024-01-01,100
1,2024-02-01,200
2,2024-01-05,300


In [0]:
%sql
SELECT user_id, txn_date, amount
FROM (
    SELECT 
        user_id,
        txn_date,
        amount,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY txn_date DESC
        ) AS rn
    FROM transactions
) t
WHERE rn = 1;

user_id,txn_date,amount
1,2024-02-01,200
2,2024-01-05,300


CASE STUDY 4: 
Funnel Analysis

You have table funnel with column names – user_id and step (e.g., view, cart and purchase)


Table structure:
(1, 'view'),(1, 'cart'),(1, 'purchase'),(2, 'view'),(2, 'cart')


Problem – Count users who completed all steps (view -> cart -> purchase)


In [0]:
%sql
CREATE TABLE funnel (
    user_id INT,
    step STRING
);

In [0]:
%sql
INSERT INTO funnel VALUES
(1, 'view'),
(1, 'cart'),
(1, 'purchase'),
(2, 'view'),
(2, 'cart');

num_affected_rows,num_inserted_rows
5,5


In [0]:
%sql
select * from funnel;

user_id,step
1,view
1,cart
1,purchase
2,view
2,cart


In [0]:
%sql
SELECT COUNT(DISTINCT user_id) AS completed_users
FROM funnel
WHERE user_id IN (
    SELECT user_id
    FROM funnel
    GROUP BY user_id
    HAVING 
        COUNT(DISTINCT step) = 3
);

completed_users
1
